In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"WITHIN_REGION"
print(f"Base directory for data: {BASE_DIR}")

## Within region modelling:

### Read stakes datasets:

In [ ]:
EXPERIMENT_REGION_GROUPS = {
    "CEU": ["FR", "CH", "IT_AT"],
    "USCA": ["ALA", "CAW"],
}

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
df_all = pd.concat(dfs.values(), ignore_index=True)

df_unique = (
    df_all[["RGIId", "RGI_REGION"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

out_path = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_all_regions.csv")
df_unique.to_csv(out_path, index=False)
print(f"Saved {len(df_unique)} unique RGI IDs to {out_path}")
df_unique.head()

In [ ]:
# run it
summarize_and_plot_all_regions(dfs)

In [ ]:
# run it
plot_mb_distributions_all_regions(dfs)

### Separate into train/test glaciers:
Separate using MMD2 distance.

In [ ]:
dfs_grouped = {
    "NOR": dfs["08"],
    "ISL": dfs["06"],
    "CEU": dfs["11"],
    "ALA": dfs["01"],
    "CAW": dfs["02"],
}

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
}

for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

MONTHLY_COLS = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
    "ELEVATION_DIFFERENCE",
]
STATIC_COLS = ["aspect", "slope", "svf"]
FEATURE_COLS = MONTHLY_COLS + STATIC_COLS

In [ ]:
# A bit "dirty coding" but I want the monthly dataframes for each whole region
# so it's easier to run like this:
PAIRS = [("ISL", "NOR"), ("NOR", "ISL"), ("ISL", "CEU"), ("ISL", "ALA"),
         ("ISL", "CAW")]
PAIR_KEYS = [
    "ISL_to_region", "NOR_to_ISL", "ISL_to_CEU", "ISL_to_ALA", "ISL_to_CAW"
]

REGION_GROUPS = {
    "CEU": ["FR", "CH", "IT_AT"],
}

print(f"Pairs: (N = {len(PAIRS)})", PAIRS)
print("Pair keys:", PAIR_KEYS)

# load all stake dfs once
dfs = {
    rid: load_stakes_for_rgi_region(cfg, rid)
    for rid in tqdm(RGI_REGIONS.keys(), desc="Loading stake regions")
}

# recompute only pairs involving these symbolic/original region labels
# examples:
#   None          -> load all
#   {"CEU"}       -> recompute all pairs where src=="CEU" or tgt=="CEU"
#   {"ISL"}       -> recompute all pairs involving ISL
#   {"CEU","ISL"} -> recompute all pairs involving either CEU or ISL
#   "ALL"         -> recompute all
RECOMPUTE_REGIONS = None

all_pair_res = {}
all_pair_split_info = {}


def resolve_region(code, region_groups):
    """Return either the raw code or the grouped list of codes."""
    return region_groups.get(code, code)


def should_recompute_pair(src, tgt, recompute_regions=None):
    """
    Decide whether a pair should be recomputed.

    Parameters
    ----------
    src, tgt : str
        Pair labels as they appear in PAIRS, e.g. "CH", "ISL", "CEU".
    recompute_regions : set[str] | None | str
        - None  -> load all pairs
        - "ALL" -> recompute all pairs
        - set   -> recompute only pairs where src or tgt is in this set
    """
    if recompute_regions is None:
        return False
    if recompute_regions == "ALL":
        return True
    return (src in recompute_regions) or (tgt in recompute_regions)


for src, tgt in tqdm(PAIRS, desc="Preparing pairwise monthlies"):
    key = f"{src}_to_{tgt}"

    src_codes = resolve_region(src, REGION_GROUPS)
    tgt_codes = resolve_region(tgt, REGION_GROUPS)

    recompute_this = should_recompute_pair(
        src=src,
        tgt=tgt,
        recompute_regions=RECOMPUTE_REGIONS,
    )

    res_xreg, split_info = prepare_monthly_df_xreg_pairwise(
        cfg=cfg,
        dfs=dfs,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        source_code=src_codes,
        target_code=tgt_codes,
        run_flag=recompute_this,  # True = recompute, False = load
        region_name=f"XREG_{src}_TO_{tgt}",
        csv_subfolder=f"CrossRegional/{src}_to_{tgt}/csv",
    )

    all_pair_res[key] = res_xreg
    all_pair_split_info[key] = split_info

    df_train = res_xreg["df_train"]
    df_test = res_xreg["df_test"]

    status = "recomputed" if recompute_this else "loaded"

    print("\n" + "=" * 80)
    print(f"Pair: {key} [{status}]")
    print(f"Train glaciers ({src}):", len(split_info["train_glaciers"]))
    print(f"Test glaciers ({tgt}):", len(split_info["test_glaciers"]))
    print("Train rows:", len(df_train), "| Test rows:", len(df_test))

In [ ]:
gl_splits = {}
TEST_GLACIERS_BY_CODE = {}
for pair in all_pair_res.keys():
    src_code, tgt_code = pair.split("_to_")
    print(
        f"\n{'='*80}\nProcessing pair: {pair} (source: {src_code}, target: {tgt_code})"
    )

    df_test_region = all_pair_res[pair]["df_test"]

    best_split, best_score = None, np.inf

    for s in range(100):
        split = split_pool_holdout(
            df_region=df_test_region,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            holdout_frac=0.20,
            seed=s,
        )

        actual_frac = split["actual_holdout_frac"]
        if not (0.15 <= actual_frac <= 0.35):
            continue

        score = abs(split["mmd2_holdout_vs_region"]) + abs(
            split["mmd2_pool_vs_region"])
        if score < best_score:
            best_score = score
            best_split = split

    # --- build dataframes ---
    holdout_glaciers = best_split["holdout_glaciers"]
    pool_glaciers = best_split["pool_glaciers"]

    df_holdout = df_test_region[df_test_region["GLACIER"].isin(
        holdout_glaciers)].copy()
    df_pool = df_test_region[df_test_region["GLACIER"].isin(
        pool_glaciers)].copy()

    print(
        f"Holdout : {len(holdout_glaciers)} glaciers, {len(df_holdout)} rows")
    print(f"Pool    : {len(pool_glaciers)} glaciers, {len(df_pool)} rows")
    print(
        f"Holdout fraction (rows) : {len(df_holdout) / len(df_test_region):.1%}"
    )
    print(
        f"MMD²(holdout, Region_all)   : {best_split['mmd2_holdout_vs_region']:.4f}"
    )
    print(
        f"MMD²(pool,    Region_all)   : {best_split['mmd2_pool_vs_region']:.4f}"
    )
    print(f"\nHoldout glaciers : {holdout_glaciers}")
    print(f"Pool glaciers    : {pool_glaciers}")

    gl_splits[tgt_code] = [holdout_glaciers, pool_glaciers]
    TEST_GLACIERS_BY_CODE[tgt_code] = holdout_glaciers

In [ ]:
TEST_GLACIERS_BY_CODE

### Monthly datasets:

In [ ]:
def merge_monthly_results(res_parts, group_key=""):
    """
    Merge several monthly result dicts by concatenating shared DataFrame values
    row-wise and keeping the first non-DataFrame metadata value.
    """
    res_parts = [r for r in res_parts if r is not None]
    if len(res_parts) == 0:
        raise ValueError(f"No results to merge for {group_key}")

    merged = {}

    common_keys = set(res_parts[0].keys())
    for r in res_parts[1:]:
        common_keys &= set(r.keys())

    for k in sorted(common_keys):
        vals = [r[k] for r in res_parts]

        if all(isinstance(v, pd.DataFrame) for v in vals):
            merged[k] = pd.concat(vals, axis=0, ignore_index=True)
        elif all(v is None for v in vals):
            merged[k] = None
        else:
            merged[k] = copy.deepcopy(
                next((v for v in vals if v is not None), None))

    return merged

In [ ]:
# Example:
res_all = prepare_monthly_dfs_for_all_regions(
    cfg=cfg,
    dfs=dfs_grouped,
    paths=paths,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    run_flag=True,
    test_glaciers_by_code=TEST_GLACIERS_BY_CODE,
    region_groups=EXPERIMENT_REGION_GROUPS,
    # only_codes=["ISL", "NOR", "CEU"])
)

# For the grouped RGI regions, we can merge the results of their components:
res_all["USCA"] = merge_monthly_results(
    [
        res_all["ALA"],
        res_all["CAW"],
    ],
    group_key="USCA",
)

# Optional: rerun parts only
"""
Example: Only recompute Norway
res_all = prepare_monthlies_for_all_regions(
    cfg=cfg,
    dfs=dfs,
    paths=paths,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    run_flag=True,
    only_codes=["NOR"],   # only Norway recomputes
)"""

#### Feature overlap:

In [ ]:
# Example usage:
# res_all is what you got from prepare_monthlies_for_all_regions(...)
figs = plot_overlap_for_all_results(
    results_dict=res_all,
    cfg=cfg,
    STATIC_COLS=STATIC_COLS,
    MONTHLY_COLS=MONTHLY_COLS,
    n_iter=1000,
)

In [ ]:
figs_kde = plot_feature_overlap_all_regions(res_all, STATIC_COLS, MONTHLY_COLS)

## LSTM model
### LSTM datasets:

In [ ]:
res_all_for_lstm = {
    "NOR": res_all["NOR"],
    "ISL": res_all["ISL"],
    "CEU": res_all["CEU"],
    "USCA": res_all["USCA"],
}

logs_cache_dir = BASE_DIR / "LSTM_cache"
logs_cache_dir.mkdir(parents=True, exist_ok=True)

lstm_assets = build_or_load_lstm_all(
    cfg=cfg,
    res_all=res_all_for_lstm,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=logs_cache_dir,
    force_recompute=True,
    include_atomic=True,
    include_groups=False,
)

### LSTM parameters:

In [ ]:
dataset_keys = build_dataset_keys(
    RGI_REGIONS=RGI_REGIONS,
    experiment_region_groups=EXPERIMENT_REGION_GROUPS,
    include_atomic=True,
    include_groups=True,
    group_key_mode="code",  # → gives "CEU", "USCA"
)

print(dataset_keys)

GS_RESULTS_DIR = Path(cfg.dataPath) / path_cache / "AdapterLSTM_7/GS_results"

log_path_gs_results = {
    "ISL":
    GS_RESULTS_DIR / 'lstm_param_search_progress_OOS_ISL_2026-02-11.csv',
    "NOR":
    GS_RESULTS_DIR / 'lstm_param_search_progress_OOS_NOR_2026-02-09.csv',
    "FR": GS_RESULTS_DIR / 'lstm_param_search_progress_OOS_FR_2026-02-06.csv',
    "CH": GS_RESULTS_DIR / 'lstm_param_search_progress_region_2026-02-18.csv',
    "CEU": GS_RESULTS_DIR / 'lstm_param_search_progress_region_2026-02-18.csv',
}

default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
    experiment_region_groups=EXPERIMENT_REGION_GROUPS,
    select_by="avg_test_loss",
    include_atomic=True,
    include_groups=True,
    group_key_mode="code",
)

print(sorted(params_by_key.keys()))

In [ ]:
params_by_key = {
    'ALA': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 128,
        'num_layers': 1,
        'bidirectional': False,
        'dropout': 0.0,
        'static_layers': 1,
        'static_hidden': 128,
        'static_dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'CAW': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 128,
        'num_layers': 1,
        'bidirectional': False,
        'dropout': 0.0,
        'static_layers': 1,
        'static_hidden': 128,
        'static_dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'ISL': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 128,
        'num_layers': 1,
        'bidirectional': False,
        'dropout': 0.0,
        'static_layers': 1,
        'static_hidden': 128,
        'static_dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'SJM': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 128,
        'num_layers': 1,
        'bidirectional': False,
        'dropout': 0.0,
        'static_layers': 1,
        'static_hidden': 128,
        'static_dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'NOR': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 96,
        'num_layers': 2,
        'bidirectional': False,
        'dropout': 0.1,
        'static_layers': 1,
        'static_hidden': 32,
        'static_dropout': 0.1,
        'lr': 0.0005,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'CH': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 160,
        'num_layers': 2,
        'bidirectional': False,
        'dropout': 0.3,
        'static_layers': 2,
        'static_hidden': 32,
        'static_dropout': 0.1,
        'lr': 0.002,
        'weight_decay': 0.0001,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'FR': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 96,
        'num_layers': 2,
        'bidirectional': False,
        'dropout': 0.3,
        'static_layers': 2,
        'static_hidden': 64,
        'static_dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.05,
        'loss_spec': None
    },
    'IT_AT': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 128,
        'num_layers': 1,
        'bidirectional': False,
        'dropout': 0.0,
        'static_layers': 1,
        'static_hidden': 128,
        'static_dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'CEU': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 160,
        'num_layers': 2,
        'bidirectional': False,
        'dropout': 0.3,
        'static_layers': 2,
        'static_hidden': 32,
        'static_dropout': 0.1,
        'lr': 0.002,
        'weight_decay': 0.0001,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    },
    'USCA': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 128,
        'num_layers': 1,
        'bidirectional': False,
        'dropout': 0.0,
        'static_layers': 1,
        'static_hidden': 128,
        'static_dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 1e-05,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    }
}

### Train model:

In [ ]:
# models, infos = train_within_region_models_all(
#     cfg=cfg,
#     lstm_assets_by_key=lstm_assets,
#     params_by_key=params_by_key,
#     device=device,
#     epochs=150,
# )

# Only retrain Norway, load the rest:
models, infos = train_within_region_models_all(
    cfg=cfg,
    lstm_assets_by_key=lstm_assets,
    params_by_key=params_by_key,
    device=device,
    # train_keys=["CEU", "ISL", "NOR"],
    epochs=150,
    force_retrain=True,
    models_dir=BASE_DIR / "models",
)

### Evaluate on test:

In [ ]:
df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
    cfg=cfg,
    models_by_key=models,
    lstm_assets_by_key=lstm_assets,
    device=device,
    save_dir="figures/eval_within_region",
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(2, 2),
    grid_figsize=(18, 12))

display(df_metrics)